In [1]:
!pip install yt-dlp pandas

# Scraper tahap 1 Menggunakan keyword
Output: json platform, judul, link, creator, tanggal upload, views, like, jumlah komentar

In [12]:
import yt_dlp
import json
import os
import re
from datetime import datetime

def bersihkan_nama_folder(nama):
    nama_bersih = re.sub(r'[\\/*?:"<>|]', "", nama)
    return nama_bersih.strip()

def scrape_youtube_metadata(keyword, max_results=5):
    ydl_opts = {
        'quiet': True,              
        'extract_flat': False,      
        'ignoreerrors': True,       
        'no_warnings': True
    }

    search_query = f"ytsearch{max_results}:{keyword}"
    timestamp = datetime.now().isoformat()
    data_list = []
    
    folder_induk = f"data/youtube_{keyword.replace(' ', '_')}_assets"

    print(f"\n{'='*50}")
    print(f"Mencari {max_results} video untuk keyword: '{keyword}'...")
    print(f"{'='*50}")

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            result = ydl.extract_info(search_query, download=False)
            
            if 'entries' in result:
                for index, video in enumerate(result['entries'], start=1):
                    if not video: continue
                    
                    raw_date = video.get('upload_date', '')
                    formatted_date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}" if len(raw_date) == 8 else raw_date

                    judul_asli = video.get('title')
                    judul_aman = bersihkan_nama_folder(judul_asli)
                    path_video = os.path.join(folder_induk, judul_aman).replace("\\", "/")
                    
                    # 1. Mengekstrak ID Unik Video
                    video_id = video.get('id')

                    item = {
                        "scraped_at": timestamp,
                        "platform": "YouTube",
                        "keyword_pencarian": keyword,
                        "video_id": video_id,  # <-- ID Unik disimpan di sini
                        "judul": judul_asli,
                        "related_link": video.get('webpage_url') or f"https://www.youtube.com/watch?v={video_id}",
                        "creator": video.get('uploader'),
                        "tanggal_upload": formatted_date,
                        "views": video.get('view_count', 0),
                        "likes": video.get('like_count', 0),
                        "jumlah_komentar": video.get('comment_count', 0),
                        "lokasi_penyimpanan": path_video
                    }
                    data_list.append(item)
                    print(f"[{index}/{max_results}] ✔️ Berhasil: {judul_asli[:40]}...")
        except Exception as e:
            print(f"Error saat scraping: {e}")

    return data_list

# ==========================================
# EKSEKUSI TAHAP 1
# ==========================================
daftar_keyword = ["kodashop"] # Bisa ditambah
jumlah_target_per_keyword = 15

os.makedirs("data", exist_ok=True)

for kw in daftar_keyword:
    hasil = scrape_youtube_metadata(kw, max_results=jumlah_target_per_keyword)

    if hasil:
        filename = f"data/youtube_{kw.replace(' ', '_')}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(hasil, f, indent=4, ensure_ascii=False)
        print(f"✅ Metadata '{kw}' disimpan di: {filename}")



Mencari 15 video untuk keyword: 'kodashop'...


ERROR: [youtube] dlhLBf_RIr4: This video is not available
ERROR: [youtube] g6xexGJlkTg: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] eGgAhlpuR8E: This video is not available
ERROR: [youtube] fHC2EOz3PrA: This video is not available
ERROR: [youtube] aNad2Ml2Lfw: This video is not available


[1/15] ✔️ Berhasil: Codashop...
[2/15] ✔️ Berhasil: [DANCE PRACTICE] SAJA BOYS 사자 보이즈 - SODA...
[3/15] ✔️ Berhasil: Saja Boys x DEKSORKRAO - Soda Pop | K-Po...
[4/15] ✔️ Berhasil: Saja Boys - Soda Pop (Official Lyric Vid...
[5/15] ✔️ Berhasil: 【INDO COVER】SODA POP - SAJA BOYS (KPOP D...
[6/15] ✔️ Berhasil: 【INDO COVER】FREE (KPOP Demon Hunters) VI...
[7/15] ✔️ Berhasil: BORONG ES KRIM SAJA BOYS KPOP DEMON HUNT...
[8/15] ✔️ Berhasil: M/V 'SODA POP - SAJA BOYS (KPOP DEMON HU...
[9/15] ✔️ Berhasil: 【INDO COVER】YOUR IDOL - SAJA BOYS (KPOP ...
[11/15] ✔️ Berhasil: CARA TOP UP ROBLOX DI CODASHOP | CARA ME...
[12/15] ✔️ Berhasil: Demon Hunters KPop 2025 dari Lagu & Suar...
[13/15] ✔️ Berhasil: Tebak Karakter Film KPop Demon Hunters 2...
[14/15] ✔️ Berhasil: CARA TOP UP DIAMOND MOBILE LEGENDS DI CO...
[15/15] ✔️ Berhasil: RESMI KERJASAMA ❗ CARA TOP UP COIN GAME ...
✅ Metadata 'kodashop' disimpan di: data/youtube_kodashop.json


# Scarping Komentar berdasarkan link pada json sebelumnya
Output: json, Judul, komentar, nama akun yang komentar

In [5]:
import json
import yt_dlp
import os

file_video_sumber = "data/youtube_balap_kuda.json"

def scrape_komentar_ke_folder(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        video_data = json.load(f)

    ydl_opts_comments = {
        'quiet': True,
        'extract_flat': False,
        'skip_download': True,    
        'getcomments': True,      
        'extractor_args': {'youtube': {'comment_sort': ['top'], 'max_comments': ['20,all,all']}},
        'ignoreerrors': True,
        'no_warnings': True
    }

    print(f"Memulai scraping komentar dan menyebarnya ke folder masing-masing...\n")

    with yt_dlp.YoutubeDL(ydl_opts_comments) as ydl:
        for index, video in enumerate(video_data, start=1):
            url = video.get('related_link')
            judul = video.get('judul')
            lokasi_folder = video.get('lokasi_penyimpanan')
            
            print(f"[{index}/{len(video_data)}] Memproses: {judul[:40]}...")
            
            # 1. Pastikan foldernya dibuat
            os.makedirs(lokasi_folder, exist_ok=True)
            
            try:
                info = ydl.extract_info(url, download=False)
                comments = info.get('comments', [])
                
                if not comments:
                    print("  -> ⚠️ Tidak ada komentar.")
                    continue
                
                hasil_komentar_video_ini = []
                
                for c in comments:
                    id_komentar = c.get('id')
                    id_parent = c.get('parent')
                    
                    # Logika Reply
                    tipe_komentar = "Komentar Utama" if (id_parent == 'root' or id_parent is None) else "Balasan"

                    komentar_item = {
                        "id_komentar": id_komentar,
                        "id_parent": id_parent,
                        "tipe": tipe_komentar,
                        "nama_channel": c.get('author'),
                        "isi_komentar": c.get('text')
                    }
                    hasil_komentar_video_ini.append(komentar_item)
                
                # 2. Simpan file komentar.json khusus di dalam folder video tersebut
                file_komentar = os.path.join(lokasi_folder, "komentar.json")
                with open(file_komentar, "w", encoding="utf-8") as f:
                    json.dump(hasil_komentar_video_ini, f, indent=4, ensure_ascii=False)
                
                print(f"  -> ✔️ Tersimpan {len(comments)} komentar di: {file_komentar}")
                
            except Exception as e:
                print(f"  -> ❌ Error: {e}")

# ==========================================
# EKSEKUSI TAHAP 2
# ==========================================
if os.path.exists(file_video_sumber):
    scrape_komentar_ke_folder(file_video_sumber)
    print("\n✅ Proses ekstraksi komentar ke masing-masing folder selesai!")
else:
    print(f"File {file_video_sumber} tidak ditemukan.")

Memulai scraping komentar dan menyebarnya ke folder masing-masing...

[1/14] Memproses: FULL RACE PACUAN KUDA PANGANDARAN 2025...
  -> ✔️ Tersimpan 21 komentar di: data/youtube_balap_kuda_assets/FULL RACE PACUAN KUDA PANGANDARAN 2025\komentar.json
[2/14] Memproses: Kentucky Derby 2023 (FULL RACE) | NBC Sp...
  -> ✔️ Tersimpan 1405 komentar di: data/youtube_balap_kuda_assets/Kentucky Derby 2023 (FULL RACE)  NBC Sports\komentar.json
[3/14] Memproses: WHR's Ten Best Races of 2024! Ft. DO DEU...
  -> ✔️ Tersimpan 36 komentar di: data/youtube_balap_kuda_assets/WHR's Ten Best Races of 2024! Ft. DO DEUCE and VIA SISTINA!\komentar.json
[4/14] Memproses: kuda Nakal Paling Belakang 🐎 Juara 1. in...
  -> ✔️ Tersimpan 227 komentar di: data/youtube_balap_kuda_assets/kuda Nakal Paling Belakang 🐎 Juara 1. indahnya Dunia Pacuan Kuda\komentar.json
[5/14] Memproses: KUDA SEHARGA LAMBORGINI! 25 Jenis Kuda T...
  -> ✔️ Tersimpan 146 komentar di: data/youtube_balap_kuda_assets/KUDA SEHARGA LAMBORGINI! 25 J

# Scarping download video berdasarkan link hasil scarp keyword

In [9]:
import json
import yt_dlp
import os

file_video_sumber = "data/youtube_balap_kuda.json"

def download_video_ke_folder(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        video_data = json.load(f)

    print(f"Memulai proses download untuk {len(video_data)} video...\n")

    for index, video in enumerate(video_data, start=1):
        url = video.get('related_link')
        judul = video.get('judul')
        lokasi_folder = video.get('lokasi_penyimpanan')
        
        # Pastikan folder sudah ada (jaga-jaga jika skip tahap 2)
        os.makedirs(lokasi_folder, exist_ok=True)
        
        print(f"\n{'='*60}")
        print(f"[{index}/{len(video_data)}] Mengunduh ke: {lokasi_folder}")
        print(f"{'='*60}")
        
        # Konfigurasi diset dinamis agar 'outtmpl' mengarah ke folder yang tepat
        ydl_opts_download = {
            'format': 'best', # Ambil yang gambar+suara sudah menyatu
            'outtmpl': f'{lokasi_folder}/video_lengkap.%(ext)s', # Nama file diseragamkan
            'quiet': False, 
            'ignoreerrors': True,
            'no_warnings': True
        }
        
        with yt_dlp.YoutubeDL(ydl_opts_download) as ydl:
            try:
                ydl.download([url])
            except Exception as e:
                print(f"❌ Gagal mengunduh {judul}: {e}")

# ==========================================
# EKSEKUSI TAHAP 3
# ==========================================
if os.path.exists(file_video_sumber):
    download_video_ke_folder(file_video_sumber)
    print(f"\n✅ Proses unduh selesai! Semua file telah masuk ke foldernya masing-masing.")
else:
    print(f"File {file_video_sumber} tidak ditemukan.")

Memulai proses download untuk 14 video...


[1/14] Mengunduh ke: data/youtube_balap_kuda_assets/FULL RACE PACUAN KUDA PANGANDARAN 2025
[youtube] Extracting URL: https://www.youtube.com/watch?v=qCiI30IVsaw
[youtube] qCiI30IVsaw: Downloading webpage
[youtube] qCiI30IVsaw: Downloading android vr player API JSON
[info] qCiI30IVsaw: Downloading 1 format(s): 18
[download] Destination: data\youtube_balap_kuda_assets\FULL RACE PACUAN KUDA PANGANDARAN 2025\video_lengkap.mp4
[download] 100% of  259.23MiB in 00:00:22 at 11.77MiB/s     

[2/14] Mengunduh ke: data/youtube_balap_kuda_assets/Kentucky Derby 2023 (FULL RACE)  NBC Sports
[youtube] Extracting URL: https://www.youtube.com/watch?v=aUDgaN6iHFc
[youtube] aUDgaN6iHFc: Downloading webpage
[youtube] aUDgaN6iHFc: Downloading android vr player API JSON
[info] aUDgaN6iHFc: Downloading 1 format(s): 18
[download] Destination: data\youtube_balap_kuda_assets\Kentucky Derby 2023 (FULL RACE)  NBC Sports\video_lengkap.mp4
[download] 100% of   23.91MiB in

# FINAL GABUNGAN

In [ ]:
import yt_dlp
import json
import os
import time

# ==========================================
# KONFIGURASI PROTOTYPE
# ==========================================
FILE_KEYWORD_INPUT = "data/input_keywords.json" 
JUMLAH_VIDEO_PER_KEYWORD = 10                 
DOWNLOAD_VIDEO_FISIK = True                 

# ==========================================
# FUNGSI 1: SCRAPING METADATA (FILTER KATA PER KATA)
# ==========================================
def scrape_youtube_metadata(keyword, max_results=5):
    ydl_opts = {
        'quiet': True,              
        'extract_flat': False,      
        'ignoreerrors': True,       
        'no_warnings': True
    }

    limit_pencarian = max_results * 3 
    search_query = f"ytsearch{limit_pencarian}:{keyword}"
    timestamp = datetime.now().isoformat()
    data_list = []
    
    folder_induk = f"data/youtube_{keyword.replace(' ', '_')}_assets"
    os.makedirs(folder_induk, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"🎬 TAHAP 1: MENCARI VIDEO RELEVAN UNTUK '{keyword}'")
    print(f"{'='*60}")

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            result = ydl.extract_info(search_query, download=False)
            
            if result and 'entries' in result:
                valid_count = 0 
                
                # ----------------------------------------------------
                # PERSIAPAN FILTER: Memecah keyword menjadi list kata
                # Contoh: "kasus cyberbullying" -> ["kasus", "cyberbullying"]
                # ----------------------------------------------------
                kata_kunci_list = keyword.lower().split()
                
                for video in result['entries']:
                    if not video: continue
                    if valid_count >= max_results: break
                        
                    judul_asli = video.get('title', 'Tanpa Judul')
                    deskripsi = video.get('description', '')
                    creator = video.get('uploader', '')
                    
                    # Gabungkan semua teks video menjadi satu blok teks panjang
                    teks_gabungan = f"{judul_asli} {deskripsi} {creator}".lower()
                    
                    # ----------------------------------------------------
                    # LOGIKA FILTER BARU: Cek apakah ada kata yang cocok
                    # ----------------------------------------------------
                    lolos_filter = False
                    for kata in kata_kunci_list:
                        # Abaikan kata sambung pendek (seperti 'di', 'ke', 'dan')
                        # Cek apakah kata kunci (panjang > 2) ada di teks gabungan
                        if len(kata) > 2 and kata in teks_gabungan:
                            lolos_filter = True
                            break # Cukup 1 kata cocok, langsung loloskan!
                    
                    if not lolos_filter:
                        print(f"  -> 🚫 SKIP (Tidak Relevan): {judul_asli[:30]}...")
                        continue 
                    
                    valid_count += 1
                    raw_date = video.get('upload_date', '')
                    formatted_date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}" if len(raw_date) == 8 else raw_date
                    video_id = video.get('id')
                    
                    path_video = os.path.join(folder_induk, str(video_id)).replace("\\", "/")
                    os.makedirs(path_video, exist_ok=True) 

                    item = {
                        "scraped_at": timestamp,
                        "platform": "YouTube",
                        "keyword_pencarian": keyword,
                        "video_id": video_id,
                        "judul": judul_asli,
                        "related_link": video.get('webpage_url') or f"https://www.youtube.com/watch?v={video_id}",
                        "creator": creator,
                        "tanggal_upload": formatted_date,
                        "views": video.get('view_count'),
                        "likes": video.get('like_count', 0) if video.get('like_count') is not None else 0,
                        "jumlah_komentar": video.get('comment_count', 0) if video.get('comment_count') is not None else 0,
                        "caption": deskripsi,
                        "Path": path_video
                    }
                    data_list.append(item)
                    
                    file_metadata_lokal = f"{path_video}/metadata.json"
                    with open(file_metadata_lokal, "w", encoding="utf-8") as f:
                        json.dump(item, f, indent=4, ensure_ascii=False)

                    print(f"[{valid_count}/{max_results}] ✔️ TERSIMPAN (Folder: {video_id})")
                
                if valid_count < max_results:
                    print(f"⚠️ PENCARIAN DIHENTIKAN: Hanya menemukan {valid_count} video relevan.")

        except Exception as e:
            print(f"❌ Error saat scraping metadata: {e}")

    return data_list

# ==========================================
# FUNGSI 2: SCRAPING KOMENTAR
# ==========================================
def scrape_komentar_ke_folder(video_data_list): # Menerima list data di memori
    ydl_opts_comments = {
        'quiet': True, 'extract_flat': False, 'skip_download': True, 
        'getcomments': True, 'extractor_args': {'youtube': {'comment_sort': ['top'], 'max_comments': ['30,all,all']}},
        'ignoreerrors': True, 'no_warnings': True
    }

    print(f"\n💬 TAHAP 2: KOMENTAR")
    with yt_dlp.YoutubeDL(ydl_opts_comments) as ydl:
        for index, video in enumerate(video_data_list, start=1):
            video_id = video.get('video_id')
            lokasi = video.get('Path') # Membaca key 'Path'
            
            print(f"[{index}/{len(video_data_list)}] Menarik komentar dari ID: {video_id}...")
            
            try:
                info = ydl.extract_info(video_id, download=False)
                if not info: continue
                comments = info.get('comments', [])
                if not comments: 
                    print("  -> ⚠️ Tidak ada komentar / Dinonaktifkan.")
                    continue
                
                hasil_komentar = [{"video_id": video_id, "id_komentar": c.get('id'), "id_parent": c.get('parent'), "tipe": "Komentar Utama" if (c.get('parent') == 'root' or c.get('parent') is None) else "Balasan", "nama_channel": c.get('author'), "isi_komentar": c.get('text')} for c in comments]
                
                with open(os.path.join(lokasi, "komentar.json"), "w", encoding="utf-8") as f:
                    json.dump(hasil_komentar, f, indent=4, ensure_ascii=False)
                print(f"  -> ✔️ Berhasil menyimpan {len(comments)} komentar.")
            except:
                print("  -> ❌ Gagal menarik komentar.")

# ==========================================
# FUNGSI 3: DOWNLOAD VIDEO (BISA DIMATIKAN)
# ==========================================
def download_video_ke_folder(video_data_list): # Menerima list data di memori
    if not DOWNLOAD_VIDEO_FISIK:
        print("\n⏭️ TAHAP 3: UNDUH VIDEO DILOMPATI (Set DOWNLOAD_VIDEO_FISIK = True untuk mengunduh)")
        return

    print(f"\n⬇️ TAHAP 3: UNDUH VIDEO FISIK")
    for index, video in enumerate(video_data_list, start=1):
        url = video.get('related_link')
        lokasi = video.get('Path') # Membaca key 'Path'
        
        print(f"[{index}/{len(video_data_list)}] Mengunduh video ke folder {lokasi}...")
        
        ydl_opts_dl = {'format': 'best', 'outtmpl': f'{lokasi}/video_lengkap.%(ext)s', 'quiet': True, 'ignoreerrors': True, 'no_warnings': True}
        with yt_dlp.YoutubeDL(ydl_opts_dl) as ydl:
            try: 
                ydl.download([url])
                print("  -> ✔️ Download berhasil.")
            except: 
                print("  -> ❌ Gagal mengunduh.")

# ==========================================
# EKSEKUSI PROTOTYPE (PIPELINE IN-MEMORY)
# ==========================================
from datetime import datetime

# 1. Buat file dummy output model jika belum ada
os.makedirs(os.path.dirname(FILE_KEYWORD_INPUT), exist_ok=True)
if not os.path.exists(FILE_KEYWORD_INPUT):
    with open(FILE_KEYWORD_INPUT, "w", encoding="utf-8") as f:
        json.dump(["kodashop", "coklat putih"], f, indent=4)

# 2. Baca keyword
with open(FILE_KEYWORD_INPUT, "r", encoding="utf-8") as f:
    daftar_keyword_dari_model = json.load(f)

print(f"🚀 MEMULAI PROTOTYPE CRAWLER...")

# 3. Eksekusi Berurutan tanpa Index File
for kw in daftar_keyword_dari_model:
    # list_data_video akan berisi LIST of DICTIONARY dari Tahap 1
    list_data_video = scrape_youtube_metadata(kw, max_results=JUMLAH_VIDEO_PER_KEYWORD)
    
    # Jika list tidak kosong, langsung oper list tersebut ke fungsi selanjutnya
    if list_data_video:
        scrape_komentar_ke_folder(list_data_video)
        download_video_ke_folder(list_data_video)

print("\n🏁 PROTOTYPE SELESAI DIEKSEKUSI!")

🚀 MEMULAI PROTOTYPE CRAWLER...

🎬 TAHAP 1: MENCARI VIDEO RELEVAN UNTUK 'kasus cyberbullying anak'


ERROR: [youtube] rRFKxtMd--E: This video is not available
ERROR: [youtube] kMQzcsaRtyk: This video is not available


[1/10] ✔️ TERSIMPAN (Folder: roQNP7S3udU)
[2/10] ✔️ TERSIMPAN (Folder: EtOO_XRT2qc)
[3/10] ✔️ TERSIMPAN (Folder: gcPa418mAdI)
[4/10] ✔️ TERSIMPAN (Folder: Xc0zAM8ehCM)
[5/10] ✔️ TERSIMPAN (Folder: KMFFFIjtEPQ)
[6/10] ✔️ TERSIMPAN (Folder: MGTrgBwXEpk)
[7/10] ✔️ TERSIMPAN (Folder: NOZsmUjxCLw)
[8/10] ✔️ TERSIMPAN (Folder: MvXBjWir03c)
[9/10] ✔️ TERSIMPAN (Folder: UL0RbQ98xH8)
[10/10] ✔️ TERSIMPAN (Folder: Ch7AqRyN5k8)

💬 TAHAP 2: KOMENTAR
[1/10] Menarik komentar dari ID: roQNP7S3udU...
  -> ✔️ Berhasil menyimpan 1 komentar.
[2/10] Menarik komentar dari ID: EtOO_XRT2qc...
  -> ⚠️ Tidak ada komentar / Dinonaktifkan.
[3/10] Menarik komentar dari ID: gcPa418mAdI...
  -> ⚠️ Tidak ada komentar / Dinonaktifkan.
[4/10] Menarik komentar dari ID: Xc0zAM8ehCM...
  -> ⚠️ Tidak ada komentar / Dinonaktifkan.
[5/10] Menarik komentar dari ID: KMFFFIjtEPQ...
  -> ✔️ Berhasil menyimpan 18 komentar.
[6/10] Menarik komentar dari ID: MGTrgBwXEpk...
  -> ✔️ Berhasil menyimpan 11 komentar.
[7/10] Menarik kome

ERROR: [youtube] WBjx4oFpQwk: This video is not available


[1/10] ✔️ TERSIMPAN (Folder: ykorFERNDWg)
[2/10] ✔️ TERSIMPAN (Folder: pKr_Hn8snfk)
[3/10] ✔️ TERSIMPAN (Folder: bqWpg3lPjQs)
[4/10] ✔️ TERSIMPAN (Folder: j5IyNOMW_Ek)
[5/10] ✔️ TERSIMPAN (Folder: -On-R4yXZ9Y)
[6/10] ✔️ TERSIMPAN (Folder: 0PM-tolXLn4)
[7/10] ✔️ TERSIMPAN (Folder: zAc-Nz6TMm0)
[8/10] ✔️ TERSIMPAN (Folder: CbVwNh9b8AM)
[9/10] ✔️ TERSIMPAN (Folder: bglc226dhWY)
[10/10] ✔️ TERSIMPAN (Folder: 7bN4laaeGGw)

💬 TAHAP 2: KOMENTAR
[1/10] Menarik komentar dari ID: ykorFERNDWg...
  -> ⚠️ Tidak ada komentar / Dinonaktifkan.
[2/10] Menarik komentar dari ID: pKr_Hn8snfk...
  -> ✔️ Berhasil menyimpan 3 komentar.
[3/10] Menarik komentar dari ID: bqWpg3lPjQs...
  -> ✔️ Berhasil menyimpan 2 komentar.
[4/10] Menarik komentar dari ID: j5IyNOMW_Ek...
  -> ⚠️ Tidak ada komentar / Dinonaktifkan.
[5/10] Menarik komentar dari ID: -On-R4yXZ9Y...
  -> ⚠️ Tidak ada komentar / Dinonaktifkan.
[6/10] Menarik komentar dari ID: 0PM-tolXLn4...
  -> ✔️ Berhasil menyimpan 1 komentar.
[7/10] Menarik koment

ERROR: [youtube] rRFKxtMd--E: This video is not available
ERROR: [youtube] Z_rG3aY2e98: This video is not available
ERROR: [youtube] XU4Cq8aWz6Q: This video is not available


[1/10] ✔️ TERSIMPAN (Folder: GheF6UT7-TA)
[2/10] ✔️ TERSIMPAN (Folder: E13Qf4VSd-0)
[3/10] ✔️ TERSIMPAN (Folder: Xc0zAM8ehCM)
[4/10] ✔️ TERSIMPAN (Folder: ay_Yg9TASEY)
[5/10] ✔️ TERSIMPAN (Folder: ENZFT5_ZC2g)
[6/10] ✔️ TERSIMPAN (Folder: tVTU0Xkuk6s)
[7/10] ✔️ TERSIMPAN (Folder: cxPFgHZw8PY)
[8/10] ✔️ TERSIMPAN (Folder: IIxbc8lBtfY)
[9/10] ✔️ TERSIMPAN (Folder: Ej87IFvvr1Q)
[10/10] ✔️ TERSIMPAN (Folder: KnW0t16ulAs)

💬 TAHAP 2: KOMENTAR
[1/10] Menarik komentar dari ID: GheF6UT7-TA...
  -> ⚠️ Tidak ada komentar / Dinonaktifkan.
[2/10] Menarik komentar dari ID: E13Qf4VSd-0...
  -> ✔️ Berhasil menyimpan 1 komentar.
[3/10] Menarik komentar dari ID: Xc0zAM8ehCM...
  -> ⚠️ Tidak ada komentar / Dinonaktifkan.
[4/10] Menarik komentar dari ID: ay_Yg9TASEY...
  -> ⚠️ Tidak ada komentar / Dinonaktifkan.
[5/10] Menarik komentar dari ID: ENZFT5_ZC2g...
  -> ✔️ Berhasil menyimpan 1 komentar.
[6/10] Menarik komentar dari ID: tVTU0Xkuk6s...
  -> ✔️ Berhasil menyimpan 33 komentar.
[7/10] Menarik komen

ERROR: [youtube] yiKeLOKc1tw: This video is not available
ERROR: [youtube] PYqqAt_VTFA: This video is not available


[1/10] ✔️ TERSIMPAN (Folder: bumpRLyF_Gc)
[2/10] ✔️ TERSIMPAN (Folder: 4OtnOqeksBw)
[3/10] ✔️ TERSIMPAN (Folder: qNskX8A5I90)
[4/10] ✔️ TERSIMPAN (Folder: _fl7wWbV604)
[5/10] ✔️ TERSIMPAN (Folder: u_vVcm6Ijyk)
[6/10] ✔️ TERSIMPAN (Folder: ThCcmEbBLc8)
[7/10] ✔️ TERSIMPAN (Folder: pfSNq9wTgYI)
[8/10] ✔️ TERSIMPAN (Folder: IvBfxSJWx6A)
[9/10] ✔️ TERSIMPAN (Folder: rHB4MZSXj6M)
[10/10] ✔️ TERSIMPAN (Folder: 851JUJyAwNc)

💬 TAHAP 2: KOMENTAR
[1/10] Menarik komentar dari ID: bumpRLyF_Gc...
  -> ✔️ Berhasil menyimpan 8 komentar.
[2/10] Menarik komentar dari ID: 4OtnOqeksBw...
  -> ✔️ Berhasil menyimpan 1 komentar.
[3/10] Menarik komentar dari ID: qNskX8A5I90...
  -> ✔️ Berhasil menyimpan 138 komentar.
[4/10] Menarik komentar dari ID: _fl7wWbV604...
  -> ✔️ Berhasil menyimpan 1465 komentar.
[5/10] Menarik komentar dari ID: u_vVcm6Ijyk...
  -> ✔️ Berhasil menyimpan 1 komentar.
[6/10] Menarik komentar dari ID: ThCcmEbBLc8...
  -> ✔️ Berhasil menyimpan 8 komentar.
[7/10] Menarik komentar dari ID